# RNN의 한계

## 기존 모델은 **컨텍스트 벡터**의 크기가 고정되어 있으며 디코더는 이 컨텍스트 벡터 하나를 이용해 출력문장을 생성하기에 문장이 길어질수록 정보 손실의 문제가 발생

더하여 
1) 마지막으로 갈수록 마지막 단어의 정보에 비해 앞쪽 단어 정보의 영향력이 희미해지고

2) 각 출력 단어마다 필요한 정보를 다르게 반영할 수 없다는 단점이 존재

=> 이를 해결하기위해 디코더가 단어를 생성할 때 입력 문장 **전체**를 다시보고 인코더와 디코더의 모든 위치를 직접 비교하고, 단어와 관련있는 부분에 집중에 **출력단어마다** 새로운 컨텍스트 벡터를 만드는 attention이 등장!

## attention
- 현재 시점에서 가장 중요한 부분에 더 높은 가중치를 부여하는 기술

- 출력 단어를 생성할 때 입력 문장 속 단어들 중 어떤 단어에 더 집중할지의 정도를 결정

- 모든 위치를 직접 비교해 병렬화가 가능하지만, 여전히 인코더와 디코더가 LSTM으로 한칸씩 순서대로 처리되기에 모델 전체의 병렬화가 이루어지지는 않음 

### attention - 과정
1. 입력을 벡터 형태로 변환
2. 관련도 점수와 softmax를 통해 단어별 중요도 계산
3. 어텐션 가중치만큼 섞어서 정보 조합

### attention - 장점
- 모든 단어를 직접 비교해 거리에 상관없이 관계 파악
- 모든 위치를 직접 연결해 한번에 전역적인 관계 학습
- 관련도 계산은 각 단어 쌍마다 서로 독립적이기에 동시에 계산 가능한 병렬화

## Attention is All You Need
> 순차구조를 없애서 병렬화를 통해 빠른 학습이 가능하다!

- 순차 구조로 인한 한계로 한 문장 안에서 병렬처리가 불가능한 RNN기반 모델들의 한계
- RNN과 CNN을 완전히 제거해 self-attention으로 입력과 출력의 관계를 계산하는 transformer모델 제안
- 이것이 LLM과 NLP의 근간이 됨

### Transformer
> attention만으로 문장 전체의 의미와 구조를 파악하여 한번에 처리하는 인코더-디코더 모델

기존 모델과의 차이점: 인코더&디코더의 내부를 무엇으로 채웟는가

-> LSTM이 아닌 **Self-Attention!**

-> attention이 모델 그 자체가 됨

- 핵심 기술
    - self-attention: 문장 안 단어들이 서로를 참고하여 중요한 관계성을 파악
    - Multi-Head attention: attention을 여러갈래로 나눠서 다양한 관점에서 관계 파악
    - positional encoding: 각 단어의 위치 정보를 더해 순서 정보를 보완




# Self-Attetion

## Query, Key, Value
- Query
    - 현재 처리중인 단어가 다른 단어들에게 지금 나에게 필요한 정보가 무엇인지 묻는 벡터

- Key
    - 각 단어가 자신이 가지고 있는 정보를 보여주는 벡터
    - 키는 검색 대상이 되는 모든 정보의 이름표 

- Value
    - 각 단어다 실제로 전달하는 정보벡터
    - 관련도 비율만큼 모든 value를 섞어 최종 결과를 만듦

## Cross-Attention vs. Self-Attention
1. **Cross-Attention**: 번역 중인 문장이 원문을 참고
- 디코더의 쿼리로 인코더의 키, 벨류 조회하고 유사도를 계산하며 가중치를 섞어 컨텍스트 벨류 생성
- 쿼리와 키-벨류가 서로 다른 시퀀스에서 출력
- 원문 내부의 관계는 RNN이 순서대로 읽어 간접적으로 파악하고 병렬처리가 불가함

2. **Self-Attention**: 문장이 스스로 내부 관계 파악
- 같은 문장에서 만든 Q, K, V로 모든 단어끼리 비교
- 쿼리와 키-벨류 쌍이 같은 시퀀스에서 출력
- 문맥이 반영된 새 표현이 결과벡터가 됨

## Scaled Dot-Product Attention
> 쿼리와 키를 비교하는 방법

### Dot-Product Attention
쿼리와 키 벡터를 두고 두 벡터의 같은 위치 성분끼리 곱해 모두 더한 값(내적)으로 관련도 점수를 구함

- 과정
    1. attention score 계산
    - 쿼리와 전치한 키를 곱하는 행렬곱셈 한번으로 점수를 동시에 계산 (병렬처리)
    2. 계산된 값에 softMax 적용해서 attention weight 산출
    - 이 단어는 다른 단어들을 몇 %씩 참고하는가를 나타냄
    3. 최종 결과값 계산
    - attention weight에다가 value를 곱하기

=> 한계: 벡터차원이 커지면 내적 값이 지나치게 커질 수 있음. 그러면 소프트멕스 함수에서 특정 단어에 확률이 과도하게 몰려 기울기가 0이 되고 학습이 원활하게 진행되지 않음

### Scaled Dot-Product Attention
> 내적과 softmax 사이에 스케일링 단계 하나를 추가해 위 한계를 해결

- 위에 2단계 전 Attention Score를 루트 디 케이로 나누는 스케일링 과정 진행
- 이 과정을 통해 차원에 상관없이 점수가 안정적인 범위에 머물 수 있음





# Multi-Head Attention
>head란 attention 계산을 한번 하는 독립된 단위

각 헤드마다 가중치 행렬의 숫자가 다르기 때문에 같은 문장을 봐도 집중하는 단어가 달라짐

## Single-Head Attention: 헤드가 1개

- 각 단어의 백터에 가중치 행렬을 **한 세트**만 곱해서 쿼리랑 키를 만든 후, 루트 디로 나눠 소프트 맥스를 적용하여 가중치 분포를 얻고 벨류에 섞어서 새 백터 하나 출력!

- 한계: 헤드가 하나이기 때문에 이 하나로 여러 관계에 골고루 가중치를 나누어주기에 결과는 여러정보가 섞여 흐릿해진 벡터 하나가 됨



## Multi-Head Attention: attention을 작게 쪼개서 여러번 동시에 함

- 각 벡터에 헤드마다 다른 가중치 행렬세트를 곱해서 64차원 쿼리, 키, 벨류를 8세트 만듦

- 8개의 헤드가 간섭하지 않고 각자의 attention을 계산, 8개의 64차원 결과를 이어붙여 다시 512차원으로 만든 후 또 하나의 행렬을 곱해서 8개의 관점을 하나로 섞음

- 즉, 헤드마다 질문 하나에 집중해서 결과를 나란히 이어붙이기에 네 정보가 섞이지 않고 또렷하게 남음

## Multi-Head Attention 작동방식

1. 512차원 벡터 하나에 헤드마다 다른 가중치 행렬을 곱해 64차원짜리 쿼리, 키, 벨류를 8세트 만듦
2. 8개의 헤드가 서로 간섭하지 않고 동시에 병렬 어텐션 계산
3. 8개의 64차원 결과를 순서대로 이어붙여 512차원 벡터 만듦
4. 이 벡터에 또 하나의 행렬을 곱해 8개 관점을 하나의 의미로 융합

# Transformer 전체 아키텍처

### 트랜스포머의 전처리 단계
1. 텍스트 입력
2. 텍스트 토큰화: 문장을 기본단위로 자르는 것
3. 토큰 임베딩: 각 토큰을 숫자(벡터)로 바꾸는 것
    - 임베딩이란 단어를 공간적 의미로 표현하는 것!
    - 뜻이 비슷한 단어는 가까운 좌표에 놓임
4. positional Encoding: 단어들의 순서 정보(위치벡터)를 임베딩 벡터에 추가
    - transformer는 모든 단어를 한꺼번에 보기에 순서 정보를 따로 알려줘야 함

### Encoder와 Decoder
- encoder
    - Multi-Head Self-Attention: 원문 단어끼리 서로를 보며 문맥파악
    - Feed-Forward Network: 단어 하나하나를 따로 더 깊이 가공(단어마다 따로, 같은 가중치로 적용)
    - Add: 부품을 통과한 결과에 원래 입력을 그대로 더함 (잔차연결)
    - Norm: 백터 안의 512개 숫자를 평균 0, 분산 1 비슷하게 맞춰줌 (층 정규화) 

- decoder
    - Masked Multi-Head Self-Attention: 미래 시점의 정보를 가려 지금까지 쓴 번역 단어끼리 관계 파악
    - Encoder-Decoder Attention: 출력 문장과 입력문장을 연결해 원분 분석 자료
    - Feed-Forward Network: Encoder와 동일
